# E01 · Frozen test inference

Chỉ inference từ R01 v1/S01 v2 tự train scratch. Bundle không chứa gold test. Chưa chạy: xem STATUS.json và RESUME.md.


In [ ]:
from pathlib import Path
import os, sys, subprocess
import torch
print("torch:",torch.__version__,"device:",torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
files=list(Path("/kaggle/input").rglob("infer_frozen.py"))
if len(files)!=1:
    raise RuntimeError("Attach exactly one frozen inference bundle")
code_dir=files[0].parent
bundle=code_dir.parent
output=Path("/kaggle/working/E01")
command=[sys.executable,str(files[0]),"--bundle",str(bundle),"--out",str(output)]
result=subprocess.run(command,env={**os.environ,"PYTHONPATH":str(code_dir)},text=True,check=True,capture_output=True)
print(result.stdout)


## Dev reproduction

So sánh các metric này với report đã lưu trước khi chấm test cục bộ. Không dùng test để đổi recipe.


In [ ]:
for task,model in [(1,"R01"),(1,"C02"),(1,"C03"),(2,"S01"),(3,"S01")]:
    result=subprocess.run([sys.executable,str(bundle/"metrics_subtask_1_2_3.py"),"-t",str(task),"-g",str(bundle/"data"/f"eng_restaurant_dev_task{task}.jsonl"),"-p",str(output/f"dev_task{task}_{model}.jsonl")],capture_output=True,text=True,check=True)
    (output/f"dev_task{task}_{model}_official_scorer.log").write_text(result.stdout)
    print(task,model,result.stdout)
print((output/"report.json").read_text())
